In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlproyectomarlon")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/items.csv"

In [0]:
df_items = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
items_schema = StructType(fields=[StructField("Item Code", StringType(), False),
                                     StructField("Item Name", StringType(), True),
                                     StructField("Category Code", IntegerType(), True),
                                     StructField("Category Name", StringType(), True)
])

In [0]:
df_items_final = spark.read\
.option('header', True)\
.schema(items_schema)\
.csv(ruta)

In [0]:
items_selected_df = df_items_final.select(col("Item Code"), 
                                                col("Item Name"), 
                                                col("Category Code"), 
                                                col("Category Name"))

In [0]:
items_renamed_df = items_selected_df.withColumnRenamed("Item Code", "ItemCode") \
                                    .withColumnRenamed("Item Name", "ItemName") \
                                    .withColumnRenamed("Category Code", "CategoryCode") \
                                    .withColumnRenamed("Category Name", "CategoryName") 

In [0]:
df_items_final = items_renamed_df

In [0]:
df_items_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.items")

In [0]:
df_items_final.show(10, truncate=False)